In [ ]:
import numpy as np
import netCDF4 as nc
from scipy.interpolate import griddata
import scipy.io as sio

In [ ]:
path = '<path-to-folder>'
datapath = 'f{path}/data/'

In [ ]:
bm3 = datapath + 'BedMachineAntarctica-v3.nc'

ds_geom = nc.Dataset(bm3)

x_g = ds_geom['x'][:]
y_g = ds_geom['y'][:]

In [ ]:
### creat a from smb and dHdt, interpolate on BedMap grid

Xs,Ys = np.meshgrid(x_g,y_g)
# Xs,Ys = np.meshgrid(x_v,y_v)

# get DHdt

ddH = nc.Dataset('data/itslive_data/ANT_G1920_GroundedIceHeight_v01.nc')

# dHdtmask = np.flipud(nc.Dataset(f'domains/{domain}_dHdt_mask.nc')['Band1'][:]).flatten()
# dHdtmask = np.where(dHdtmask>0)

dHJan18 = ddH.variables['height_change'][393,:,:].flatten()#[dHdtmask]
dHJan13 = ddH.variables['height_change'][333,:,:].flatten()#[dHdtmask]

xdH, ydH = np.meshgrid(ddH.variables['x'][:], ddH.variables['y'][:])
XdH = xdH.flatten()#[dHdtmask]
YdH = ydH.flatten()#[dHdtmask]

fiveyrs = 31536000.0 * 5. + 60.*60.*24. # leap year in 2016
dH1318 = dHJan18 - dHJan13
dHdt1318 = dH1318 / fiveyrs

dHmask = np.where(dHdt1318 > -30_000)

dHdt1318 = dHdt1318[dHmask]
XdH = XdH[dHmask]
YdH = YdH[dHmask]

# get smb

dsmb = nc.Dataset('data/SMB_Noell2023/smb_rec.1979-2021.RACMO2.3p2_ANT27_ERA5-3h.AIS.2km.YY.nc')

# average smb for 2013 to 2017 (5 yrs)
smb = dsmb.variables['smb_rec'][34,:,:].flatten()#[smbmask]
for i in np.arange(35,39):
    smb += dsmb.variables['smb_rec'][i,:,:].flatten()#[smbmask]
smb = smb/5.

# convert from mm w.e./yr to m i.e./s
smb = smb * 10**-3 * 1.091 * 31536000.0**-1

xsmb, ysmb = np.meshgrid(dsmb.variables['x'][:], dsmb.variables['y'][:])
Xsmb = xsmb.flatten()#[smbmask]
Ysmb = ysmb.flatten()#[smbmask]

# interpolate

dH_interp = griddata((XdH,YdH),dHdt1318,(Xs,Ys),method='cubic')
smb_interp = griddata((Xsmb,Ysmb),smb,(Xs,Ys),method='cubic')

# compute a

a = smb_interp - dH_interp

In [ ]:
data = {"a-2013-17": a, "x": Xs, "y": Ys}
sio.savemat(f"MC-SSA/domains/apparent_mass_balance_2013-17_mps_masked_grounded.mat", data)